# 가상 대학 푸드코트: 누적 학습과 독립 평가
실제 대학의 공개 메뉴·운영자료를 참고한 가상 환경입니다. 실제 학교의 성과 예측이 아닙니다. CPU 병렬 실험이며 GPU를 선택해도 자동 가속되지 않습니다. 먼저 pilot을 실행하세요.

코랩 연결이 종료되면 마지막 완료 세대부터 재개합니다. 갑작스러운 종료로 worker.lock이 남으면 실행 중인 작업이 없는지 직접 확인한 뒤 해당 잠금 파일만 제거하세요.

In [ ]:
import os, subprocess, pathlib
def run_visible(args, cwd=None):
    with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status=process.wait()
    if status:
        raise subprocess.CalledProcessError(status,args)

os.environ["OMP_NUM_THREADS"]="1"
os.environ["OPENBLAS_NUM_THREADS"]="1"
REPOSITORY="https://github.com/phj9187/koisk-simulation.git"
REVISION="86647c15878d7c45b59fc88674d494c15e4f7a64" # 검증한 알고리즘 버전 고정
WORK=pathlib.Path("/content/foodcourt_code")
if not WORK.exists(): run_visible(["git","clone",REPOSITORY,str(WORK)])
run_visible(["git","-C",str(WORK),"fetch","origin",REVISION])
run_visible(["git","-C",str(WORK),"checkout","--detach","FETCH_HEAD"])
run_visible(["git","-C",str(WORK),"rev-parse","HEAD"])
run_visible(["pip","install","-q","-r",str(WORK/"requirements-foodcourt.txt")])
os.environ["PYTHONPATH"]=str(WORK/"src")

In [ ]:
USE_DRIVE=True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive") # 사용자가 직접 인증
BASE=pathlib.Path("/content/drive/MyDrive/kiosk_foodcourt" if USE_DRIVE else "/content/foodcourt_results")
PRESET="pilot" # colab, low_acceptance, uncertain_taste, tight_capacity, low_waste
EXPERIMENT_ID="foodcourt_pilot_001" # 설정/소스가 바뀌면 새 이름; 같은 이름으로 재개
ROOT=BASE/EXPERIMENT_ID
CONFIG=WORK/"config"/f"foodcourt_{PRESET}.yaml"
command=["python","-m","kiosk_foodcourt.cli","--config",str(CONFIG),"--root",str(ROOT)]
run_visible(command+["--plan"],cwd=WORK)

다음 셀이 실제 학습을 실행합니다. 대규모 설정은 많은 CPU 시간을 사용합니다. `MAX_JOBS`는 이번 연결에서 처리할 작업 수이며, 각 작업의 학습 상태는 이어집니다. 시간 제한은 세대 사이에 확인하므로 한 세대나 최종 평가가 끝날 때까지 초과할 수 있습니다. Drive를 끄면 런타임 종료 시 결과가 사라질 수 있습니다.

In [ ]:
MAX_JOBS=4
TIME_BUDGET_MINUTES=20
run_visible(command+["--max-jobs",str(MAX_JOBS),"--time-budget-minutes",str(TIME_BUDGET_MINUTES)],cwd=WORK)

In [ ]:
import json
summary=ROOT/"summary.json"
if summary.exists():
    rows=json.loads(summary.read_text())
    for row in rows:
        if row["policy"].startswith("learned") and row["baseline"]=="customer_only": print(row)
print("저장 위치:", ROOT)
print("표본이 적은 pilot은 실행 확인용입니다. 정확도·현실 효과의 증거가 아닙니다.")